In [ ]:
!pip install -q bitsandbytes "google-genai<3" transformers peft accelerate pydantic

import os
import re
import gc
import json
import time
import math
import torch
from typing import List, Literal
from pydantic import BaseModel, Field, field_validator
from google.colab import drive
from google import genai
from google.genai import types
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. GẮN GOOGLE DRIVE ĐỂ LƯU & KHÔI PHỤC TIẾN TRÌNH TỰ ĐỘNG
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/rsibench_colab_exp"
os.makedirs(f"{OUTPUT_DIR}/datasets", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/checkpoints", exist_ok=True)

# 2. CẤU HÌNH THỰC NGHIỆM
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"          # Điền API Key từ Google AI Studio
AGENT_MODEL_ID = "gemini-3.6-flash"             # Researcher Agent kiêm Rollout Model
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"    # Mô hình gốc M0
MAX_WALL_CLOCK_MINUTES = 18.0                   # Ngân sách cứng 18 phút

# Tập đánh giá cố định EVAL_SET (nạp từ file eval_set.jsonl trên Google Drive)
EVAL_FILE_PATH = f"{OUTPUT_DIR}/eval_set.jsonl"

DEFAULT_EVAL_SET = [
    {
        "id": 1,
        "type": "cong_nhan",
        "prompt": "5 người thợ may mất 5 phút để may được 5 chiếc áo. Hỏi 100 người thợ may mất bao lâu để may được 100 chiếc áo với cùng năng suất? Hãy suy luận và đưa ra kết quả cuối cùng.",
        "expected_keyword": "5 phút"
    },
    {
        "id": 2,
        "type": "tuoi_tac",
        "prompt": "Năm nay anh 6 tuổi và gấp đôi tuổi em. Hỏi khi anh 60 tuổi thì em bao nhiêu tuổi? Hãy suy luận và đưa ra kết quả cuối cùng.",
        "expected_keyword": "57"
    },
    {
        "id": 3,
        "type": "oc_sen",
        "prompt": "Một con ốc sên trèo lên một cái cây cao 10 mét. Ban ngày nó trèo lên được 3 mét, ban đêm nó lại bị tuột xuống 2 mét. Hỏi sau bao nhiêu ngày thì con ốc sên trèo lên đến đỉnh cây? Hãy suy luận và đưa ra kết quả cuối cùng.",
        "expected_keyword": "8"
    }
]

def load_or_init_eval_set(file_path: str) -> list:
    if not os.path.exists(file_path):
        with open(file_path, "w", encoding="utf-8") as f:
            for item in DEFAULT_EVAL_SET:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"Đã khởi tạo tệp mẫu tại: {file_path}")

    required_keys = {"id", "type", "prompt", "expected_keyword"}
    dataset = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            if not required_keys.issubset(item.keys()):
                raise ValueError(f"Dòng {line_num} thiếu trường bắt buộc: {required_keys - set(item.keys())}")
            dataset.append({
                "id": int(item["id"]),
                "type": str(item["type"]),
                "prompt": str(item["prompt"]),
                "expected_keyword": str(item["expected_keyword"]),
            })
    if not dataset:
        raise ValueError("Tập đánh giá eval_set.jsonl đang rỗng!")
    return dataset

EVAL_SET = load_or_init_eval_set(EVAL_FILE_PATH)

# Đặc tả định dạng đầu ra & Giao thức chấm điểm cố định (Scoring & I/O Protocol)
FINAL_ANSWER_PREFIX = "Đáp án cuối cùng:"
FORMAT_SUFFIX = (
    "\n\nHãy suy luận từng bước và kết thúc câu trả lời bằng dòng cuối cùng theo đúng định dạng:\n"
    f"{FINAL_ANSWER_PREFIX} <kết quả>"
)

FIXED_MAX_SEQ_LENGTH = 768  # Cố định ở tầng hạ tầng, Agent không được chỉnh

# Danh sách siêu tham số huấn luyện cho phép (Whitelisted Config c_t)
CONFIG_WHITELIST = {
    "lora_r": [4, 8, 16],
    "learning_rate": [5e-5, 1e-4, 2e-4, 3e-4],
    "num_train_epochs": [1, 2, 3]
}

client = genai.Client(api_key=GEMINI_API_KEY)

# 3. HẠ TẦNG CỐ ĐỊNH (FIXED INFRASTRUCTURE)
class FixedInfrastructure:
    def __init__(self, model_id: str):
        self.model_id = model_id
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        self.base_model = prepare_model_for_kbit_training(
            self.base_model, use_gradient_checkpointing=False
        )

    @staticmethod
    def verify_final_line(response_text: str, expected_keyword: str) -> dict:
        lines = [line.strip() for line in response_text.strip().splitlines() if line.strip()]
        if not lines:
            return {"format_valid": False, "extracted_answer": "", "passed": False}

        last_line = lines[-1]
        prefix_pattern = r"^" + re.escape(FINAL_ANSWER_PREFIX) + r"\s*(.+)$"
        match = re.match(prefix_pattern, last_line, flags=re.IGNORECASE)
        if not match:
            return {"format_valid": False, "extracted_answer": last_line, "passed": False}

        extracted = match.group(1).strip()
        kw_pattern = r"(?<!\d)" + re.escape(expected_keyword.strip()) + r"(?!\d)"
        passed = bool(re.search(kw_pattern, extracted, flags=re.IGNORECASE))
        return {"format_valid": True, "extracted_answer": extracted, "passed": passed}

    def evaluate(self, model) -> dict:
        model.eval()
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

        results = []
        correct = 0
        for item in EVAL_SET:
            wrapped_prompt = item["prompt"] + FORMAT_SUFFIX
            messages = [{"role": "user", "content": wrapped_prompt}]
            prompt_text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = self.tokenizer(prompt_text, return_tensors="pt").to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=384,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id
                )

            gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
            response_text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            verdict = self.verify_final_line(response_text, item["expected_keyword"])
            if verdict["passed"]:
                correct += 1

            results.append({
                "id": item["id"],
                "type": item["type"],
                "expected_keyword": item["expected_keyword"],
                "format_valid": verdict["format_valid"],
                "extracted_answer": verdict["extracted_answer"],
                "passed": verdict["passed"],
                "model_output": response_text
            })

        return {
            "score": round((correct / len(EVAL_SET)) * 100, 2),
            "correct": correct,
            "total": len(EVAL_SET),
            "details": results
        }

    @staticmethod
    def verify_python_logic(code_str: str, final_line: str) -> bool:
        if not code_str or not code_str.strip():
            return False

        def _safe_import(name, *args, **kwargs):
            if name == "math":
                return math
            raise ImportError("Chỉ cho phép dùng thư viện math")

        safe_globals = {
            "__builtins__": {"__import__": _safe_import, "abs": abs, "round": round, "min": min, "max": max},
            "math": math,
        }
        local_vars = {}
        try:
            exec(code_str, safe_globals, local_vars)
        except Exception:
            return False

        val = local_vars.get("result")
        if val is None or isinstance(val, bool):
            return False

        val_str = str(int(val)) if isinstance(val, float) and val.is_integer() else str(val).strip()
        if not val_str:
            return False

        # Dùng biên số (?<!\d)...(?!\d) giống verify_final_line để tránh "8" khớp nhầm vào "18"
        kw_pattern = r"(?<!\d)" + re.escape(val_str) + r"(?!\d)"
        return bool(re.search(kw_pattern, final_line))

    def validate_and_train(self, attempt_id: int, dataset: List[dict], config: dict) -> dict:
        for key, allowed_vals in CONFIG_WHITELIST.items():
            if config.get(key) not in allowed_vals:
                return {"status": "INVALID", "error": f"Tham số {key}={config.get(key)} nằm ngoài whitelist {allowed_vals}."}

        if not dataset or len(dataset) < 1:
            return {"status": "INVALID", "error": "Tập dữ liệu D_t quá ít (cần tối thiểu 3 mẫu)."}

        eval_prompts_norm = [x["prompt"].lower().strip() for x in EVAL_SET]
        clean_samples = []
        for row in dataset:
            u_text = row.get("user_prompt", "").strip()
            a_text = row.get("assistant_response", "").strip()
            if not u_text or not a_text:
                continue
            if any(ep in u_text.lower() or u_text.lower() in ep for ep in eval_prompts_norm):
                return {"status": "INVALID", "error": "Vi phạm ràng buộc: D_t chứa nội dung trùng lặp với EVAL_SET."}

            # Lọc cứng: bỏ qua mẫu nếu lời giải không chốt đúng dòng cuối 'Đáp án cuối cùng: <kết quả>'
            last_line = [ln.strip() for ln in a_text.splitlines() if ln.strip()][-1]
            if not re.match(r"^" + re.escape(FINAL_ANSWER_PREFIX) + r"\s*.+$", last_line, flags=re.IGNORECASE):
                continue

            # Lọc cứng bằng trình thực thi Python: loại mẫu nếu tính nhẩm sai đáp án
            if not self.verify_python_logic(row.get("verify_python", ""), last_line):
                continue

            u_wrapped = u_text if FINAL_ANSWER_PREFIX in u_text else u_text + FORMAT_SUFFIX
            clean_samples.append((u_wrapped, a_text))

        if len(clean_samples) < 1:
            return {"status": "INVALID", "error": "Không đủ mẫu hợp lệ sau khi lọc."}

        max_len = FIXED_MAX_SEQ_LENGTH
        tokenized_data = []
        for u_text, a_text in clean_samples:
            prompt_only = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": u_text}], tokenize=False, add_generation_prompt=True
            )
            full_conv = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": u_text}, {"role": "assistant", "content": a_text}],
                tokenize=False
            )
            prompt_ids = self.tokenizer(prompt_only, truncation=True, max_length=max_len)["input_ids"]
            full_enc = self.tokenizer(full_conv, truncation=True, max_length=max_len, return_tensors="pt")
            input_ids = full_enc["input_ids"][0]
            labels = input_ids.clone()
            labels[:min(len(prompt_ids), len(labels))] = -100
            tokenized_data.append((input_ids, labels))

        # Khởi tạo LoRA mới hoàn toàn từ mô hình gốc M0
        lora_r = config["lora_r"]
        lora_cfg = LoraConfig(
            r=lora_r,
            lora_alpha=lora_r * 2,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        peft_model = get_peft_model(self.base_model, lora_cfg)
        peft_model.train()

        optimizer = torch.optim.AdamW(peft_model.parameters(), lr=config["learning_rate"])
        accum_steps = 4
        total_loss = 0.0
        micro_steps = 0
        optimizer.zero_grad()

        for _ in range(config["num_train_epochs"]):
            for input_ids, labels in tokenized_data:
                outputs = peft_model(
                    input_ids=input_ids.unsqueeze(0).to(peft_model.device),
                    labels=labels.unsqueeze(0).to(peft_model.device)
                )
                loss = outputs.loss
                if not torch.isnan(loss):
                    (loss / accum_steps).backward()
                    total_loss += loss.item()
                    micro_steps += 1
                    if micro_steps % accum_steps == 0:
                        optimizer.step()
                        optimizer.zero_grad()

        # Cập nhật nốt phần gradient lẻ cuối cùng (nếu tổng số mẫu không chia hết cho 4)
        if micro_steps % accum_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

        eval_res = self.evaluate(peft_model)
        ckpt_path = f"{OUTPUT_DIR}/checkpoints/attempt_{attempt_id}"
        peft_model.save_pretrained(ckpt_path)

        # Tháo bỏ LoRA để khôi phục nguyên trạng M0 cho vòng lặp tiếp theo
        self.base_model = peft_model.unload()
        del peft_model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

        return {
            "status": "VALID",
            "train_samples": len(clean_samples),
            "avg_train_loss": round(total_loss / max(micro_steps, 1), 4),
            "ckpt_path": ckpt_path,
            "eval_sel": eval_res
        }

# 4. KHAI BÁO RESEARCHER AGENT (GEMINI 3.5 FLASH)
class SyntheticExample(BaseModel):
    category: str = Field(description="Nhóm kỹ năng suy luận nhắm tới")
    user_prompt: str = Field(description="Đề bài toán đố tiếng Việt mới (không sao chép bài trong tập đánh giá)")
    assistant_response: str = Field(description="Lời giải từng bước rõ ràng và kết thúc bằng dòng cuối theo đúng đặc tả I/O")
    verify_python: str = Field(
        description=(
            "Đoạn mã Python thuần tính lại đáp án số học của bài toán và gán vào biến `result` "
            "(có thể dùng thư viện `math`, không dùng `print` hay thư viện ngoài). "
            "Ví dụ cú pháp: `result = 15 * 4 + 10`"
        )
    )

class WhitelistedConfigSchema(BaseModel):
    lora_r: Literal["4", "8", "16"]
    learning_rate: Literal["5e-5", "1e-4", "2e-4", "3e-4"]
    num_train_epochs: Literal["1", "2", "3"]

    @field_validator("lora_r", "num_train_epochs", mode="after")
    @classmethod
    def _to_int(cls, v: str) -> int:
        return int(v)

    @field_validator("learning_rate", mode="after")
    @classmethod
    def _to_float(cls, v: str) -> float:
        return float(v)

class AgentTurnOutput(BaseModel):
    diagnosis: str = Field(description="Chuẩn đoán lỗ hổng năng lực hoặc lỗi định dạng của mô hình hiện tại")
    decision: Literal["CONTINUE", "STOP"] = Field(description="Quyết định tiếp tục thử nghiệm vòng mới (CONTINUE) hoặc kết thúc tìm kiếm (STOP)")
    decision_rationale: str = Field(description="Lập luận chiến lược cho quyết định CONTINUE hoặc STOP")
    selected_best_attempt: int = Field(ge=1, description="ID của checkpoint đã huấn luyện được chọn làm tốt nhất (từ 1 trở đi, tuyệt đối không chọn mô hình gốc 0)")
    hypothesis: str = Field(description="Giả thuyết nghiên cứu dữ liệu cho lần thử tiếp theo (nếu chọn CONTINUE)")
    config: WhitelistedConfigSchema
    synthetic_dataset: List[SyntheticExample] = Field(description="Tập dữ liệu huấn luyện tổng hợp D_t (12 đến 24 mẫu)")

class FinalSelectionOutput(BaseModel):
    selected_best_attempt: int = Field(ge=1, description="ID của checkpoint đã huấn luyện cuối cùng được chọn làm tốt nhất từ các attempt hợp lệ (từ 1 trở đi, tuyệt đối không chọn 0)")
    selection_rationale: str = Field(description="Lập luận chọn checkpoint cuối cùng")

SYSTEM_INSTRUCTION = f"""
Bạn là một AI Researcher Agent thực hiện quy trình RSIBench-Data (Data-Centric Post-Training Research).
Mục tiêu của bạn là thiết kế dữ liệu huấn luyện (SFT) để cải thiện một mô hình ngôn ngữ mục tiêu (M0) trên một benchmark gồm 3 dạng bài toán đố suy luận tiếng Việt:
1. Bài toán năng suất đồng thời (nhiều người cùng làm một việc trong thời gian không đổi).
2. Bài toán hiệu số tuổi không đổi theo thời gian.
3. Bài toán tiến-lùi có điểm dừng ở bước cuối cùng (vượt chướng ngại vật/leo cây).

ĐẶC TẢ MÔI TRƯỜNG & GIAO THỨC CHẤM ĐIỂM (I/O & SCORING PROTOCOL):
- Khi đánh giá, hệ thống sẽ nối thêm chỉ thị định dạng vào cuối đề bài:
  "{FORMAT_SUFFIX.strip()}"
- Bộ chấm điểm (Verifier) chỉ trích xuất duy nhất nội dung nằm sau "{FINAL_ANSWER_PREFIX}" ở dòng cuối cùng của câu trả lời để đối chiếu với đáp án chuẩn. Nếu dòng cuối cùng không tuân thủ đúng định dạng này, câu trả lời sẽ bị tính là sai (format_valid = False).
- Dữ liệu mẫu bạn tạo ra trong D_t cần đồng bộ với hành vi và định dạng đầu ra mà bộ chấm điểm yêu cầu.

RÀNG BUỘC DỮ LIỆU & HẠ TẦNG:
- Không có dữ liệu mồi (seed): Bạn tự thiết kế và sinh toàn bộ dữ liệu huấn luyện D_t (12–24 mẫu/vòng).
- Chống rò rỉ dữ liệu: Tuyệt đối không đưa nguyên văn hoặc biến thể gần giống câu chữ của các bài toán trong kết quả đánh giá vào D_t. Hãy xây dựng các bài toán mới cùng bản chất logic nhưng khác bối cảnh và số liệu.
- Mỗi vòng thử nghiệm (Attempt t), hệ thống luôn huấn luyện LoRA mới độc lập từ mô hình gốc M0 trên tập D_t và cấu hình c_t bạn đề xuất.
- Bạn toàn quyền kiểm soát chiến lược tìm kiếm trong giới hạn ngân sách thời gian: tự quyết định tiếp tục (CONTINUE) hay dừng lại (STOP), và chỉ định checkpoint tốt nhất (selected_best_attempt).
- Kiểm chứng toán học bằng code: Với mỗi mẫu trong synthetic_dataset, bắt buộc viết mã Python vào trường `verify_python` để tính độc lập kết quả số học cuối cùng và gán vào biến `result` (ví dụ cú pháp: `result = 15 * 4 + 10`), đảm bảo `result` khớp chính xác với đáp án ở dòng cuối của `assistant_response`.
"""

def format_history_for_agent(history: List[dict]) -> list:
    summary = []
    for h in history:
        summary.append({
            "attempt_id": h["attempt_id"],
            "status": h["status"],
            "hypothesis": h.get("hypothesis"),
            "config": h.get("config"),
            "train_samples": h.get("train_samples"),
            "avg_train_loss": h.get("avg_train_loss"),
            "eval_sel_score": h.get("eval_sel", {}).get("score") if h.get("eval_sel") else None,
            "eval_details": [
                {
                    "id": d["id"],
                    "type": d["type"],
                    "format_valid": d["format_valid"],
                    "extracted_answer": d["extracted_answer"],
                    "passed": d["passed"],
                    "output_snippet": d["model_output"]
                }
                for d in h.get("eval_sel", {}).get("details", [])
            ],
            "error": h.get("error")
        })
    return summary

def query_researcher_agent(base_eval: dict, history: List[dict], elapsed_min: float, max_min: float):
    user_prompt = f"""
HỢP ĐỒNG NGÂN SÁCH THỜI GIAN:
- Thời gian đã sử dụng: {elapsed_min:.2f} phút / {max_min:.2f} phút.
- Thời gian còn lại: {max(0.0, max_min - elapsed_min):.2f} phút.

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH GỐC M0 (Attempt 0):
{json.dumps(base_eval, ensure_ascii=False, indent=2)}

LỊCH SỬ CÁC LẦN THỬ TRƯỚC ĐÓ (H_<t):
{json.dumps(format_history_for_agent(history), ensure_ascii=False, indent=2)}

Dựa trên phản hồi đánh giá và ngân sách thời gian còn lại, hãy đưa ra chuẩn đoán, quyết định CONTINUE hoặc STOP, chọn checkpoint tốt nhất hiện tại, và đề xuất dữ liệu D_t cùng cấu hình c_t (nếu tiếp tục).
"""
    response = client.models.generate_content(
        model=AGENT_MODEL_ID,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=AgentTurnOutput,
            temperature=0.4,
        ),
    )
    usage = {
        "prompt_tokens": getattr(response.usage_metadata, "prompt_token_count", 0),
        "output_tokens": getattr(response.usage_metadata, "candidates_token_count", 0),
    }
    return AgentTurnOutput.model_validate_json(response.text), usage

def query_final_selection(base_eval: dict, history: List[dict]):
    user_prompt = f"""
Ngân sách thời gian thực nghiệm đã kết thúc. Dưới đây là kết quả mô hình gốc M0 (Attempt 0) và toàn bộ lịch sử các lần thử nghiệm (H_T):

KẾT QUẢ M0 (Attempt 0):
{json.dumps(base_eval, ensure_ascii=False, indent=2)}

LỊCH SỬ TOÀN BỘ CÁC LẦN THỬ (H_T):
{json.dumps(format_history_for_agent(history), ensure_ascii=False, indent=2)}

Hãy chọn ra một checkpoint duy nhất (`selected_best_attempt`) tốt nhất từ lịch sử trên.
"""
    response = client.models.generate_content(
        model=AGENT_MODEL_ID,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=FinalSelectionOutput,
            temperature=0.0,
        ),
    )
    usage = {
        "prompt_tokens": getattr(response.usage_metadata, "prompt_token_count", 0),
        "output_tokens": getattr(response.usage_metadata, "candidates_token_count", 0),
    }
    return FinalSelectionOutput.model_validate_json(response.text), usage

# 5. KHỞI TẠO HẠ TẦNG & KIỂM TRA/TẢI ĐIỂM GỐC M0 TỪ GOOGLE DRIVE
infra = FixedInfrastructure(BASE_MODEL_ID)
base_eval_path = f"{OUTPUT_DIR}/base_eval.json"

if os.path.exists(base_eval_path):
    with open(base_eval_path, "r", encoding="utf-8") as f:
        base_eval = json.load(f)
    print(f"[KHÔI PHỤC TỪ DRIVE] Đã tải kết quả điểm gốc M0 ({BASE_MODEL_ID}): {base_eval['score']}% ({base_eval['correct']}/3)")
else:
    base_eval = infra.evaluate(infra.base_model)
    with open(base_eval_path, "w", encoding="utf-8") as f:
        json.dump(base_eval, f, ensure_ascii=False, indent=2)
    print(f"=== ĐIỂM MÔ HÌNH GỐC M0 ({BASE_MODEL_ID}) ===")
    print(f"Điểm gốc M0: {base_eval['score']}% ({base_eval['correct']}/{base_eval['total']})")

for d in base_eval["details"]:
    print(f"- Bài {d['id']} ({d['type']}): {'ĐẠT' if d['passed'] else 'TRƯỢT'} | Format hợp lệ: {d['format_valid']} | Trích xuất: '{d['extracted_answer']}' (Kỳ vọng: '{d['expected_keyword']}')")

In [ ]:
log_path = f"{OUTPUT_DIR}/experiment_log.json"

# 1. KHÔI PHỤC TIẾN TRÌNH TỪ GOOGLE DRIVE (NẾU CÓ)
if os.path.exists(log_path):
    with open(log_path, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    history = saved_state.get("history", [])
    prior_elapsed_min = saved_state.get("actual_wall_clock_minutes", 0.0)
    total_api_tokens = saved_state.get("total_api_tokens", {"prompt": 0, "output": 0})
    agent_selected_id = saved_state.get("agent_selected_attempt_id", 0)
    is_finished = saved_state.get("is_finished", False)
    attempt_id = (history[-1]["attempt_id"] + 1) if history else 1
    print(f"[KHÔI PHỤC TIẾN TRÌNH] Đã tải {len(history)} vòng trước đó | Thời gian đã dùng: {prior_elapsed_min:.2f}/{MAX_WALL_CLOCK_MINUTES} phút.")
else:
    history = []
    prior_elapsed_min = 0.0
    total_api_tokens = {"prompt": 0, "output": 0}
    agent_selected_id = 0
    is_finished = False
    attempt_id = 1
    print(f"=== BẮT ĐẦU PHIÊN THỰC NGHIỆM MỚI (Ngân sách: {MAX_WALL_CLOCK_MINUTES} phút) ===\n")

def save_checkpoint_state(elapsed_total_min: float, finished: bool = False, extra_fields: dict = None):
    state = {
        "base_model": BASE_MODEL_ID,
        "agent_model": AGENT_MODEL_ID,
        "max_wall_clock_minutes": MAX_WALL_CLOCK_MINUTES,
        "actual_wall_clock_minutes": round(elapsed_total_min, 2),
        "is_finished": finished,
        "total_api_tokens": total_api_tokens,
        "base_eval": base_eval,
        "agent_selected_attempt_id": agent_selected_id,
        "history": history
    }
    if extra_fields:
        state.update(extra_fields)
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

session_start = time.time()
stopped_by_time_after_train = False

if is_finished:
    print("Phiên thực nghiệm này đã hoàn tất trước đó. (Xóa file experiment_log.json trên Drive nếu muốn chạy lại từ đầu).")
else:
    while True:
        elapsed_total_min = prior_elapsed_min + (time.time() - session_start) / 60.0
        if elapsed_total_min >= MAX_WALL_CLOCK_MINUTES:
            print(f"[HẾT NGÂN SÁCH THỜI GIAN] Đã đạt {elapsed_total_min:.2f}/{MAX_WALL_CLOCK_MINUTES} phút. Kết thúc vòng lặp.")
            break

        print(f"--- Vòng {attempt_id} (Thời gian tích lũy: {elapsed_total_min:.2f} / {MAX_WALL_CLOCK_MINUTES} phút) ---")
        try:
            agent_plan, token_usage = query_researcher_agent(
                base_eval=base_eval,
                history=history,
                elapsed_min=elapsed_total_min,
                max_min=MAX_WALL_CLOCK_MINUTES
            )
            total_api_tokens["prompt"] += token_usage["prompt_tokens"]
            total_api_tokens["output"] += token_usage["output_tokens"]
        except Exception as e:
            print(f"[LỖI API AGENT] {e}")
            save_checkpoint_state(prior_elapsed_min + (time.time() - session_start) / 60.0, finished=False)
            break

        print(f"Chuẩn đoán: {agent_plan.diagnosis}")
        print(f"Quyết định: {agent_plan.decision} | Lập luận: {agent_plan.decision_rationale}")
        agent_selected_id = agent_plan.selected_best_attempt

        if agent_plan.decision == "STOP":
            print(f">>> Researcher Agent chủ động chọn STOP tại vòng {attempt_id}. Checkpoint đề xuất: Attempt {agent_selected_id}.")
            stopped_by_time_after_train = False
            break

        if prior_elapsed_min + (time.time() - session_start) / 60.0 >= MAX_WALL_CLOCK_MINUTES - 1.0:
            print("[CẢNH BÁO THỜI GIAN] Thời gian còn lại dưới 1 phút, không đủ hoàn tất chu kỳ huấn luyện. Dừng tìm kiếm.")
            break

        cfg_dict = agent_plan.config.model_dump()
        raw_dataset = [ex.model_dump() for ex in agent_plan.synthetic_dataset]

        dataset_path = f"{OUTPUT_DIR}/datasets/attempt_{attempt_id}_data.jsonl"
        with open(dataset_path, "w", encoding="utf-8") as f:
            for row in raw_dataset:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"Giả thuyết: {agent_plan.hypothesis}")
        print(f"Cấu hình c_{attempt_id}: {cfg_dict} | Số mẫu D_{attempt_id}: {len(raw_dataset)}")

        round_start = time.time()
        train_eval_res = infra.validate_and_train(attempt_id, raw_dataset, cfg_dict)
        round_dur_sec = round(time.time() - round_start, 2)
        elapsed_after_round = prior_elapsed_min + (time.time() - session_start) / 60.0

        record = {
            "attempt_id": attempt_id,
            "elapsed_min_at_end": round(elapsed_after_round, 2),
            "round_duration_sec": round_dur_sec,
            "diagnosis": agent_plan.diagnosis,
            "decision_rationale": agent_plan.decision_rationale,
            "hypothesis": agent_plan.hypothesis,
            "config": cfg_dict,
            "dataset_path": dataset_path,
            "api_tokens": token_usage,
            **train_eval_res
        }
        history.append(record)
        stopped_by_time_after_train = True

        # Lưu tiến trình ngay lập tức lên Google Drive sau mỗi vòng
        save_checkpoint_state(elapsed_after_round, finished=False)

        if train_eval_res["status"] == "VALID":
            sel_score = train_eval_res["eval_sel"]["score"]
            sel_corr = train_eval_res["eval_sel"]["correct"]
            print(f"Kết quả Attempt {attempt_id}: Eval_sel = {sel_score}% ({sel_corr}/3) | Loss = {train_eval_res['avg_train_loss']} | Thời gian: {round_dur_sec}s [Đã lưu Drive]")
            for d in train_eval_res["eval_sel"]["details"]:
                print(f"   * Bài {d['id']}: {'ĐẠT' if d['passed'] else 'TRƯỢT'} (Format: {d['format_valid']}, Trích xuất: '{d['extracted_answer']}')")
            print()
        else:
            print(f"Attempt {attempt_id} KHÔNG HỢP LỆ: {train_eval_res['error']} [Đã lưu Drive]\n")

        attempt_id += 1

    # Nếu dừng do hết giờ ngay sau khi train xong vòng cuối, cho Agent xem kết quả vòng cuối để chốt checkpoint
    if stopped_by_time_after_train and history:
        try:
            final_sel, sel_usage = query_final_selection(base_eval, history)
            total_api_tokens["prompt"] += sel_usage["prompt_tokens"]
            total_api_tokens["output"] += sel_usage["output_tokens"]
            agent_selected_id = final_sel.selected_best_attempt
            print(f"[CHỐT CUỐI NGÂN SÁCH] Agent chọn Attempt {agent_selected_id} | Lý do: {final_sel.selection_rationale}")
        except Exception as e:
            print(f"[CẢNH BÁO CHỐT CUỐI] Giữ lựa chọn gần nhất Attempt {agent_selected_id} ({e})")

# 2. TỔNG KẾT & ĐỐI CHIẾU CHECKPOINT AGENT CHỌN VS. CHECKPOINT TỐT NHẤT THỰC TẾ
valid_map = {h["attempt_id"]: h for h in history if h["status"] == "VALID"}

# Tính checkpoint tốt nhất thực tế (Historical-best)
historical_best_id = 0
historical_best_score = base_eval["score"]
for h in history:
    if h["status"] == "VALID" and h["eval_sel"]["score"] > historical_best_score:
        historical_best_score = h["eval_sel"]["score"]
        historical_best_id = h["attempt_id"]

# Lấy điểm Eval_sel tại checkpoint mà Agent tự chọn (do giải mã tham lam tất định nên Eval_off == Eval_sel)
if agent_selected_id in valid_map:
    agent_selected_score = valid_map[agent_selected_id]["eval_sel"]["score"]
    agent_selected_valid = True
else:
    agent_selected_score = None
    agent_selected_valid = False

total_wall_min = round(prior_elapsed_min + (time.time() - session_start) / 60.0, 2)
delta_agent = round(agent_selected_score - base_eval["score"], 2) if agent_selected_valid else None
delta_hist = round(historical_best_score - base_eval["score"], 2)

summary_metrics = {
    "agent_selected_attempt_id": agent_selected_id,
    "agent_selected_valid": agent_selected_valid,
    "agent_selected_eval_score": agent_selected_score,
    "agent_selected_delta_over_base": delta_agent,
    "historical_best_attempt_id": historical_best_id,
    "historical_best_eval_score": historical_best_score,
    "historical_best_delta_over_base": delta_hist,
}
save_checkpoint_state(total_wall_min, finished=True, extra_fields=summary_metrics)

print("\n==================== TỔNG KẾT KẾT QUẢ THỰC NGHIỆM ====================")
print(f"- Điểm mô hình gốc M0 (Attempt 0):              {base_eval['score']}% ({base_eval['correct']}/3)")
if agent_selected_valid:
    print(f"- Checkpoint Agent tự chọn (Attempt {agent_selected_id}):       {agent_selected_score}% (Cải thiện: +{delta_agent}%)")
else:
    print(f"- Checkpoint Agent tự chọn (Attempt {agent_selected_id}):       KHÔNG HỢP LỆ")
print(f"- Checkpoint Tốt nhất thực tế (Attempt {historical_best_id}):    {historical_best_score}% (Cải thiện: +{delta_hist}%)")
print(f"- Tổng số lần thử đã chạy:                      {len(history)} (Hợp lệ: {len(valid_map)})")
print(f"- Tổng thời gian tích lũy:                      {total_wall_min} / {MAX_WALL_CLOCK_MINUTES} phút")
print(f"- Tổng token Gemini API:                        Prompt={total_api_tokens['prompt']}, Output={total_api_tokens['output']}")
print(f"=> Toàn bộ nhật ký và dữ liệu tổng hợp đã lưu tại: {OUTPUT_DIR}")